In [ ]:
# Live Key Results (auto-populated from metrics CSV)
import pandas as pd
from pathlib import Path
from IPython.display import display

root = Path('..').resolve()
metrics_path = root / 'outputs' / 'models' / 'subgroup_model_metrics.csv'
summary_path = root / 'outputs' / 'models' / 'plots' / 'subgroup_model_summary.csv'

print('Metrics source ->', metrics_path)
if not metrics_path.exists():
    print('No subgroup metrics found. Run `src/subgroup_modeling.py` then re-run this cell.')
else:
    dfm = pd.read_csv(metrics_path)
    # Preferred highlight order (these were used previously in the notebook)
    preferred = [('16-20', 'Female'), ('20-25', 'Male'), ('16-20', 'Male')]

    rows = []
    for age, gender in preferred:
        r = dfm[(dfm['age_range'] == age) & (dfm['gender'] == gender)]
        if not r.empty:
            rows.append(r.iloc[0])

    if rows:
        report_df = pd.DataFrame(rows)
    else:
        # fallback: show top 3 groups by size
        report_df = dfm.sort_values('n', ascending=False).head(3)

    # Select columns to display and format numeric cols
    cols = ['age_range', 'gender', 'n', 'pos', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']
    present_cols = [c for c in cols if c in report_df.columns]
    # Round numeric columns for readability
    for c in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
        if c in report_df.columns:
            report_df[c] = report_df[c].round(3)

    display(report_df[present_cols].reset_index(drop=True))

    print('\nNote: small groups (n < ~10) are unstable; rows with `cv_fallback=True` used in-sample evaluation (optimistic).')

In [ ]:
# Auto Key Results (generated from metrics CSV)
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

root = Path('..').resolve()
metrics_path = root / 'outputs' / 'models' / 'subgroup_model_metrics.csv'
summary_path = root / 'outputs' / 'models' / 'plots' / 'subgroup_model_summary.csv'

if not metrics_path.exists():
    display(Markdown('**No subgroup metrics found.** Run `src/subgroup_modeling.py` and re-run this cell.'))
else:
    dfm = pd.read_csv(metrics_path)
    dfs = pd.read_csv(summary_path) if summary_path.exists() else None

    preferred = [('16-20', 'Female'), ('20-25', 'Male'), ('16-20', 'Male')]
    rows = []
    for age, gender in preferred:
        r = dfm[(dfm['age_range'] == age) & (dfm['gender'] == gender)]
        if not r.empty:
            rows.append(r.iloc[0])

    if rows:
        report_df = pd.DataFrame(rows)
    else:
        report_df = dfm.sort_values('n', ascending=False).head(3)

    if dfs is not None:
        merged = report_df.merge(dfs[['age_range', 'gender', 'pr_auc']], on=['age_range', 'gender'], how='left')
    else:
        merged = report_df.copy()
        merged['pr_auc'] = pd.NA

    def fmt(x):
        try:
            return f"{float(x):.3f}"
        except Exception:
            return 'NA'

    lines = [
        '## Key Results (summary)',
        'Below are the most relevant subgroup metrics (auto-generated). Small groups (n < 10) are unstable and flagged.'
    ]

    for _, r in merged.iterrows():
        age_label = r.get('age_range', '')
        gender = r.get('gender', '')
        n = int(r.get('n', 0)) if pd.notna(r.get('n', None)) else 'NA'
        pos = int(r.get('pos', 0)) if pd.notna(r.get('pos', None)) else 'NA'
        roc = fmt(r.get('roc_auc', 'NA'))
        pr = fmt(r.get('pr_auc', 'NA'))
        acc = fmt(r.get('accuracy', 'NA'))
        prec = fmt(r.get('precision', 'NA'))
        rec = fmt(r.get('recall', 'NA'))
        f1 = fmt(r.get('f1', 'NA'))
        lines.append(f"- `{age_label} {gender}` — n={n}, pos={pos} — ROC AUC={roc}, PR AUC={pr}, Accuracy={acc}, Precision={prec}, Recall={rec}, F1={f1}")

    lines.append('\n**Takeaway:** Subgrouping revealed heterogeneity; treat small-n groups with caution.')
    display(Markdown('\n\n'.join(lines)))

In [ ]:
# Cell: Plot bubble charts for key summary subgroups
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
from IPython.display import Image, display

# Use absolute paths from repository root to avoid working-dir issues
REPO_ROOT = Path('..').resolve()
cleaned_path = REPO_ROOT / 'data' / 'cleaned_dataset.csv'
plots_root = REPO_ROOT / 'outputs' / 'models' / 'plots'
plots_root.mkdir(parents=True, exist_ok=True)

# Key subgroups to plot (age_range, gender)
key_subgroups = [('16-20', 'Male'), ('16-20', 'Female'), ('20-25', 'Male')]

if not cleaned_path.exists():
    print('Cleaned dataset not found at', cleaned_path)
else:
    df = pd.read_csv(cleaned_path)
    # ensure age_range exists (recreate mapping if needed)
    def age_to_range(a):
        try:
            a = float(a)
        except Exception:
            return None
        if a <= 10:
            return '0-10'
        if 10 < a <= 15:
            return '10-15'
        if 15 < a <= 20:
            return '16-20'
        if 20 < a <= 25:
            return '20-25'
        if 25 < a <= 30:
            return '26-30'
        if 30 < a <= 40:
            return '31-40'
        if 40 < a <= 50:
            return '40-50'
        if 50 < a <= 60:
            return '51-60'
        if 60 < a <= 80:
            return '61-80'
        return None

    if 'age_range' not in df.columns:
        df['age_range'] = df['age'].apply(age_to_range)

    for age_label, gender in key_subgroups:
        sub = df[(df['age_range'] == age_label) & (df['gender'].str.lower() == gender.lower())]
        outdir = plots_root / f'{age_label}_{gender}'
        outdir.mkdir(parents=True, exist_ok=True)

        plt.figure(figsize=(7, 5))
        if sub.empty:
            plt.text(0.5, 0.5, f'No data for {age_label} {gender}', ha='center')
            plt.axis('off')
            fname = outdir / 'bubble.png'
            plt.savefig(fname, bbox_inches='tight')
            plt.close()
            print('Wrote empty placeholder for', age_label, gender)
            continue

        # Use time_spent_hours (x), score_pct (y), bubble size ~ age (or constant if missing)
        x = sub['time_spent_hours'].fillna(0)
        y = sub['score_pct'].fillna(0)
        ages = sub['age'].fillna(sub['age'].median()).astype(float)
        sizes = (ages - ages.min() + 1) * 30  # scale sizes

        palette = {'Male': '#1f77b4', 'Female': '#ff7f0e'}
        color = palette.get(gender, '#2ca02c')

        plt.scatter(x, y, s=sizes, c=color, alpha=0.7, edgecolors='w')
        for i, row in sub.iterrows():
            # annotate with anonymized id (short) to avoid PII
            pid = str(row.get('participant_id', ''))
            plt.annotate(str(pid), (row.get('time_spent_hours', 0), row.get('score_pct', 0)), fontsize=8, alpha=0.85, ha='center', va='center')

        # Add best-fit line (linear) when possible
        xv = x.values
        yv = y.values
        mask = (~np.isnan(xv)) & (~np.isnan(yv))
        if mask.sum() >= 2:
            coef = np.polyfit(xv[mask], yv[mask], 1)
            x_line = np.linspace(xv[mask].min(), xv[mask].max(), 200)
            y_line = np.polyval(coef, x_line)
            plt.plot(x_line, y_line, color='k', linestyle='--', linewidth=1)
            # compute R^2
            y_pred = np.polyval(coef, xv[mask])
            ss_res = ((yv[mask] - y_pred) ** 2).sum()
            ss_tot = ((yv[mask] - yv[mask].mean()) ** 2).sum()
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
            plt.text(0.99, 0.02, f'Slope={coef[0]:.3f}, R²={r2:.2f}', ha='right', va='bottom', transform=plt.gca().transAxes, fontsize=8)
        else:
            plt.text(0.5, 0.02, 'Insufficient points for fit', ha='center', fontsize=8, transform=plt.gca().transAxes)

        plt.xlabel('Time spent online (hours)')
        plt.ylabel('Score (%)')
        plt.title(f'Bubble plot — {age_label} {gender} (n={len(sub)})')
        plt.grid(True, alpha=0.3)

        # Add explanatory caption about bubble size for new users
        fig = plt.gcf()
        fig.text(0.5, 0.01, 'Bubble size ∝ participant age (years). Larger bubbles → older participants. Sizes are scaled for visibility.', ha='center', fontsize=9)

        fname = outdir / 'bubble.png'
        plt.savefig(fname, bbox_inches='tight', dpi=150)
        plt.close()
        print('Wrote bubble plot for', age_label, gender, '->', fname)

        # display inline
        display(Image(str(fname)))

In [ ]:
# Cell: Plot bubble charts for ALL subgroups (age_range × gender)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from IPython.display import Image, display

REPO_ROOT = Path('..').resolve()
cleaned_path = REPO_ROOT / 'data' / 'cleaned_dataset.csv'
plots_root = REPO_ROOT / 'outputs' / 'models' / 'plots'
plots_root.mkdir(parents=True, exist_ok=True)

if not cleaned_path.exists():
    print('Cleaned dataset not found at', cleaned_path)
else:
    df = pd.read_csv(cleaned_path)

    # ensure age_range exists
    def age_to_range(a):
        try:
            a = float(a)
        except Exception:
            return None
        if a <= 10:
            return '0-10'
        if 10 < a <= 15:
            return '10-15'
        if 15 < a <= 20:
            return '16-20'
        if 20 < a <= 25:
            return '20-25'
        if 25 < a <= 30:
            return '26-30'
        if 30 < a <= 40:
            return '31-40'
        if 40 < a <= 50:
            return '40-50'
        if 50 < a <= 60:
            return '51-60'
        if 60 < a <= 80:
            return '61-80'
        return None

    if 'age_range' not in df.columns:
        df['age_range'] = df['age'].apply(age_to_range)

    groups = df[['age_range', 'gender']].dropna().drop_duplicates().values.tolist()
    groups = sorted(groups, key=lambda x: (str(x[0]), str(x[1])))

    for age_label, gender in groups:
        sub = df[(df['age_range'] == age_label) & (df['gender'].str.lower() == str(gender).lower())]
        outdir = plots_root / f'{age_label}_{gender}'
        outdir.mkdir(parents=True, exist_ok=True)

        plt.figure(figsize=(7, 5))
        if sub.empty:
            plt.text(0.5, 0.5, f'No data for {age_label} {gender}', ha='center')
            plt.axis('off')
            fname = outdir / 'bubble_all.png'
            plt.savefig(fname, bbox_inches='tight')
            plt.close()
            print('Wrote placeholder for', age_label, gender)
            continue

        x = sub['time_spent_hours'].fillna(0)
        y = sub['score_pct'].fillna(0)
        ages = sub['age'].fillna(sub['age'].median()).astype(float)
        sizes = (ages - ages.min() + 1) * 30

        palette = {'male': '#1f77b4', 'female': '#ff7f0e'}
        color = palette.get(str(gender).lower(), '#2ca02c')

        plt.scatter(x, y, s=sizes, c=color, alpha=0.7, edgecolors='w')
        for i, row in sub.iterrows():
            pid = str(row.get('participant_id', ''))
            plt.annotate(str(pid), (row.get('time_spent_hours', 0), row.get('score_pct', 0)), fontsize=7, alpha=0.9, ha='center', va='center')

        # Add linear best-fit when possible
        xv = x.values
        yv = y.values
        mask = (~np.isnan(xv)) & (~np.isnan(yv))
        if mask.sum() >= 2:
            coef = np.polyfit(xv[mask], yv[mask], 1)
            x_line = np.linspace(xv[mask].min(), xv[mask].max(), 200)
            y_line = np.polyval(coef, x_line)
            plt.plot(x_line, y_line, color='k', linestyle='--', linewidth=1)
            y_pred = np.polyval(coef, xv[mask])
            ss_res = ((yv[mask] - y_pred) ** 2).sum()
            ss_tot = ((yv[mask] - yv[mask].mean()) ** 2).sum()
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
            plt.text(0.99, 0.02, f'Slope={coef[0]:.3f}, R²={r2:.2f}', ha='right', va='bottom', transform=plt.gca().transAxes, fontsize=8)
        else:
            plt.text(0.5, 0.02, 'Insufficient points for fit', ha='center', fontsize=8, transform=plt.gca().transAxes)

        plt.xlabel('Time spent online (hours)')
        plt.ylabel('Score (%)')
        plt.title(f'Bubble plot — {age_label} {gender} (n={len(sub)})')
        plt.grid(True, alpha=0.3)

        # Explain bubble size for users
        fig = plt.gcf()
        fig.text(0.5, 0.01, 'Bubble size ∝ participant age (years). Larger bubbles → older participants. Sizes are scaled for visibility.', ha='center', fontsize=9)

        fname = outdir / 'bubble_all.png'
        plt.savefig(fname, bbox_inches='tight', dpi=150)
        plt.close()
        print('Wrote bubble plot for', age_label, gender, '->', fname)
        display(Image(str(fname)))

## Lesser groups — side summary
We keep the small subgroups in the analysis for transparency. The table below shows groups with small sample sizes (n < 10) or where cross-validation was not possible; their metrics are optimistic and should be treated as descriptive only.

Guidance: display these results with a clear warning, and avoid using them for deployment or strong claims. Consider collecting more data before drawing firm conclusions from these groups.

In [ ]:
# Cell X: Show 'lesser' groups (n < 10 or cv_fallback True)
import pandas as pd
from pathlib import Path
root = Path('.')
metrics_path = root / 'outputs' / 'models' / 'subgroup_model_metrics.csv'
summary_path = root / 'outputs' / 'models' / 'plots' / 'subgroup_model_summary.csv'
lesser_note = 'Groups with n < 10 or where CV fallback/in-sample evaluation was used (view with caution).'
print(lesser_note)
if metrics_path.exists():
    dfm = pd.read_csv(metrics_path)
    # Mark small groups and CV fallback
    dfm['small_n'] = dfm['n'] < 10
    # Some rows may include a 'cv_fallback' column; ensure it exists
    if 'cv_fallback' not in dfm.columns:
        dfm['cv_fallback'] = False
    lesser = dfm[dfm['small_n'] | dfm['cv_fallback']].copy()
    if lesser.empty:
        print('No lesser groups found (all groups have n >= 10 and used CV).')
    else:
        display(lesser.sort_values(['n', 'age_range', 'gender']))
else:
    print('No metrics CSV found at', metrics_path)

# If you want to save a short CSV with these lesser groups for reporting:
# lesser.to_csv(root/'outputs'/'models'/'lesser_groups_summary.csv', index=False)

## Lesser Groups (Detailed)

Below are the lesser (small / potentially unreliable) subgroup results presented in the same concise style as the Key Summary. Keep these for transparency, but treat them as descriptive only.

- **16–20 Female**:
  - **n:** 8, **positives:** 6
  - **Accuracy:** 0.75, **Precision:** 0.75, **Recall:** 1.00, **F1:** 0.857
  - **ROC AUC:** 0.75, **PR AUC:** 0.906
  - **CV used:** Yes
  - **Note:** small sample (n=8); metrics are unstable and should be interpreted cautiously.

- **20–25 Female**:
  - **n:** 7, **positives:** 4
  - **Accuracy:** 0.571, **Precision:** 0.667, **Recall:** 0.50, **F1:** 0.571
  - **ROC AUC:** 0.50, **PR AUC:** 0.667
  - **CV used:** Yes
  - **Note:** n=7; descriptive only.

- **31–40 Male**:
  - **n:** 2, **positives:** 1
  - **Accuracy:** 1.00, **Precision:** 1.00, **Recall:** 1.00, **F1:** 1.00
  - **ROC AUC:** 1.00, **PR AUC:** 1.00
  - **CV used:** No (in‑sample evaluation / `cv_fallback=True`)
  - **Note:** n=2; perfect metrics reflect overfitting — do not use for inference.

- **31–40 Female**:
  - **n:** 3, **positives:** 2
  - **Accuracy:** 1.00, **Precision:** 1.00, **Recall:** 1.00, **F1:** 1.00
  - **ROC AUC:** 1.00, **PR AUC:** 1.00
  - **CV used:** No (in‑sample evaluation / `cv_fallback=True`)
  - **Note:** n=3; overfitting likely — ignore for inference.

- **61–80 Female**:
  - **n:** 2, **positives:** 1
  - **Accuracy:** 1.00, **Precision:** 1.00, **Recall:** 1.00, **F1:** 1.00
  - **ROC AUC:** 1.00, **PR AUC:** 1.00
  - **CV used:** No (in‑sample evaluation / `cv_fallback=True`)
  - **Note:** n=2; overfitting likely — ignore for inference.

**Reporting guidance:** Include these rows in appendices and clearly label them as `n < 10` or `cv_fallback=True`. Report main findings using only groups with `n >= 10` and `cv_used=True`. If you want, I can also write these rows to `outputs/models/lesser_groups_summary.csv` for easy export.

In [ ]:
# Cell 1: Load summary tables
import pandas as pd
from pathlib import Path
root = Path('.')
metrics_path = root / 'outputs' / 'models' / 'subgroup_model_metrics.csv'
summary_path = root / 'outputs' / 'models' / 'plots' / 'subgroup_model_summary.csv'
print('metrics_path ->', metrics_path)
if metrics_path.exists():
    df_metrics = pd.read_csv(metrics_path)
    display(df_metrics)
else:
    print('No metrics CSV found at', metrics_path)

if summary_path.exists():
    df_summary = pd.read_csv(summary_path)
    display(df_summary)
else:
    print('No summary CSV found at', summary_path)

In [ ]:
# Cell 2: List generated plot files for quick review
import os
plots_root = Path('outputs')/'models'/'plots'
if plots_root.exists():
    all_files = []
    for p, dirs, files in os.walk(plots_root):
        for f in files:
            if f.lower().endswith(('.png', '.html')):
                all_files.append(os.path.join(p, f))
    print(f'Found {len(all_files)} plot files under', plots_root)
    for fp in sorted(all_files)[:40]:
        print('-', fp)
else:
    print('No plots directory at', plots_root)

# To display an image inline, you can run:
# from IPython.display import Image, display
# display(Image('outputs/models/plots/20-25_Male/roc.png'))

## Reproduce
From the project root:

```bash
python src/run_all.py
```

This rebuilds `data/cleaned_dataset.csv`, `models/` and `outputs/`. Then re-run this notebook.